# Level 4 — 심층 분석: Grad-CAM 및 효율성 Trade-off

**목표**: 모델의 *동작 원리* 를 설명하고, FPS와 정확도의 trade-off 를 정량화합니다.

리포트 필수 산출물:
1. **속성별 정규화 Confusion Matrix 3개** — best 모델 기준.
2. **Grad-CAM 패널** — 같은 이미지에 대해 3개 head 가 각각 어디를 보는지 시각화 (예: `rainy + night + city street` 인 이미지).
3. 모든 백본에 대한 **FPS vs Avg-MF1 Pareto plot**.

본 노트북은 학습 노트북이 아니라 분석 노트북이지만, wandb 가 활성화되어 있으면 confusion matrix 이미지·Grad-CAM 패널·FPS 표를 같은 프로젝트의 별도 Run 으로 업로드합니다.

In [ ]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/dahye411/2026-HYU-AUE8088-PA2.git

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed
from src.utils.transforms import eval_transform
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, average_macro_f1, CLASS_NAMES
from src.utils.efficiency import measure_fps
from src.utils.wandb_logger import WandbLogger
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.xai.gradcam import GradCAM
from src.models.vgg import VGG16
from src.models.resnet import resnet18, resnet50
from src.models.vit import vit_small_patch16_224

set_seed(42, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
logger = WandbLogger(project=WANDB_PROJECT, run_name="level4-analysis", tags=["level4", "analysis"])

In [ ]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# 분석 대상으로 사용할 best 모델 로드
val_ds = BDDAttrDataset(DATA_ROOT, "val", transform=eval_transform())
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

BEST_CKPT_PATH = "./pretrained_checkpoints/level3_focal_weather_sampler_mix.pth"

model = vit_small_patch16_224().to(device)
ckpt = torch.load(BEST_CKPT_PATH, map_location=device)
state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
model.load_state_dict(state, strict=True)
model.eval()


In [ ]:
# 속성별 정규화 Confusion Matrix 생성 및 시각화
preds, probs, targets, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(preds, targets, normalize="true")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, a in zip(axes, ATTRIBUTES):
    sns.heatmap(cms[a], annot=True, fmt=".2f", cmap="Blues", ax=ax, cbar=False,
                xticklabels=CLASS_NAMES[a], yticklabels=CLASS_NAMES[a])
    ax.set_title(f"{a}")
    ax.set_xlabel("예측 (pred)"); ax.set_ylabel("정답 (true)")
fig.tight_layout()
logger.log_image("analysis/confusion_matrices", fig)

# 속성별 개별 confusion matrix 도 분리해서 업로드
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"analysis/cm_{a}", cms[a], CLASS_NAMES[a])

In [ ]:
# 속성별 Grad-CAM. ResNet 의 경우 마지막 conv 단계를 target_layer 로 사용, ViT는 마지막 block의 norm1 token activation을 target_layer로 사용.
if hasattr(model, "layer4"):
    target_layer = model.layer4[-1]
elif hasattr(model, "blocks"):
    target_layer = model.blocks[-1].norm1
else:
    raise ValueError("Grad-CAM target_layer를 자동으로 찾을 수 없습니다.")


def class_idx(attr, name):
    return CLASS_NAMES[attr].index(name)


# 리포트 해석이 쉬운 조건을 우선 선택합니다. 없으면 해당 조건은 건너뜁니다.
PREFERRED_GRADCAM_QUERIES = [
    {"weather": "rainy", "timeofday": "night", "scene": "city street"},
    {"weather": "rainy", "timeofday": "night"},
    {"weather": "snowy"},
    {"weather": "foggy"},
    {"timeofday": "dawn/dusk"},
    {"scene": "highway", "timeofday": "daytime"},
]


def matches_query(sample, query):
    for attr, name in query.items():
        if getattr(sample, attr) != class_idx(attr, name):
            return False
    return True


selected_indices = []
for query in PREFERRED_GRADCAM_QUERIES:
    for idx, sample in enumerate(val_ds.samples):
        if idx in selected_indices:
            continue
        if matches_query(sample, query):
            selected_indices.append(idx)
            break
    if len(selected_indices) >= 6:
        break

# 조건 샘플이 6장보다 적으면 validation set 앞쪽에서 중복 없이 채웁니다.
for idx in range(len(val_ds)):
    if len(selected_indices) >= 6:
        break
    if idx not in selected_indices:
        selected_indices.append(idx)

selected = [val_ds[i] for i in selected_indices[:6]]
x = torch.stack([item["image"] for item in selected]).to(device)
image_ids = [item["image_id"] for item in selected]
label_texts = [
    f"{CLASS_NAMES['weather'][item['weather'].item()]} / "
    f"{CLASS_NAMES['scene'][item['scene'].item()]} / "
    f"{CLASS_NAMES['timeofday'][item['timeofday'].item()]}"
    for item in selected
]

print("Grad-CAM samples:")
for image_id, label in zip(image_ids, label_texts):
    print(f"  {image_id}: {label}")

gc = GradCAM(model, target_layer)

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for row, attr in enumerate(ATTRIBUTES):
    # 각 속성 head 의 최대 logit 합을 score 로 사용 → 해당 head 가 "보는" 영역 추출
    cam = gc(x, lambda out, a=attr: out[a].max(dim=-1).values.sum())
    for col in range(6):
        img = x[col].detach().cpu().permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        axes[row, col].imshow(img)
        axes[row, col].imshow(cam[col].detach().cpu().numpy(), cmap="jet", alpha=0.45)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(label_texts[col], fontsize=8)
        if col == 0:
            axes[row, col].set_ylabel(attr, fontsize=14)
fig.tight_layout()
logger.log_image("analysis/gradcam_panel", fig)


In [ ]:
# FPS 측정 — 배치 1, 224x224, warm-up 20회 후 200회 평균. 모든 백본에 대해 실행.
from pathlib import Path
import pandas as pd

PARETO_SPECS = [
    {"name": "vgg16", "label": "VGG-16", "fn": VGG16, "ckpt": Path("./pretrained_checkpoints/level1_vgg16_w111.pth")},
    {"name": "resnet18", "label": "ResNet-18", "fn": resnet18, "ckpt": Path("./pretrained_checkpoints/level1_resnet18_w111.pth")},
    {"name": "resnet50", "label": "ResNet-50", "fn": resnet50, "ckpt": Path("./pretrained_checkpoints/level1_resnet50_w111.pth")},
    {"name": "vit_s16_pretrained", "label": "ViT-S/16 pretrained", "fn": vit_small_patch16_224, "ckpt": Path("./pretrained_checkpoints/level2_vit_s16_pretrained.pth")},
    {"name": "level3_best", "label": "Level 3 best", "fn": vit_small_patch16_224, "ckpt": Path("./pretrained_checkpoints/level3_focal_weather_sampler_mix.pth")},
]


def best_avg_mf1(path):
    if not path.exists():
        print(f"checkpoint 없음: {path}")
        return None
    ckpt = torch.load(path, map_location="cpu")
    hist = ckpt.get("history") if isinstance(ckpt, dict) else None
    if not hist or "val_avg_mf1" not in hist:
        print(f"history 없음: {path}")
        return None
    return float(max(hist["val_avg_mf1"]))


records = []
fps_cache = {}

for spec in PARETO_SPECS:
    arch_name = spec["fn"].__name__
    if arch_name not in fps_cache:
        m = spec["fn"]().to(device).eval()
        fps_cache[arch_name] = measure_fps(m, device, batch_size=1, n_warmup=20, n_iter=200)
        m.to("cpu")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    avg_mf1 = best_avg_mf1(spec["ckpt"])
    if avg_mf1 is None:
        continue

    fps = round(fps_cache[arch_name], 2)
    print(f"{spec['label']:24s} FPS = {fps:.2f}, best Avg-MF1 = {avg_mf1:.4f}")
    records.append({
        "backbone": spec["name"],
        "label": spec["label"],
        "fps": fps,
        "avg_mf1": avg_mf1,
        "checkpoint": str(spec["ckpt"]),
    })

fps_rows = [[r["label"], r["fps"], r["avg_mf1"], r["checkpoint"]] for r in records]
logger.log_table("analysis/fps", ["backbone", "FPS", "best_avg_mf1", "checkpoint"], fps_rows)

pareto_df = pd.DataFrame(records)
display(pareto_df)

if len(pareto_df) > 0:
    pareto_df = pareto_df.sort_values("fps")
    frontier = []
    best_so_far = -1.0
    for _, row in pareto_df.iterrows():
        if row["avg_mf1"] >= best_so_far:
            frontier.append(row)
            best_so_far = row["avg_mf1"]
    frontier_df = pd.DataFrame(frontier)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(pareto_df["fps"], pareto_df["avg_mf1"], s=80)
    for _, row in pareto_df.iterrows():
        ax.annotate(row["label"], (row["fps"], row["avg_mf1"]), textcoords="offset points", xytext=(6, 5), fontsize=9)

    if len(frontier_df) > 0:
        ax.plot(frontier_df["fps"], frontier_df["avg_mf1"], linestyle="--", marker="o", label="Pareto frontier")

    ax.set_xlabel("FPS on T4, batch=1")
    ax.set_ylabel("Best Val Avg-Macro-F1")
    ax.set_title("FPS vs Avg-Macro-F1 Pareto Plot")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    logger.log_image("analysis/pareto", fig)
    plt.show()


In [ ]:
logger.finish()